<div style="font-size:12pt; font-weight:bold;">Modularity Vitality Tests</div>
<div style="font-size:12pt;">Jonathan H. Morgan, Ph.D.</div>
<div style="font-size:12pt;">31 October 2025</div>

<div style="font-size:10pt; font-weight:bold;">Preamble</div>

In [1]:
import os
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Sequence, Mapping,  Any
import igraph as ig
from scipy.sparse import csr_matrix
from scipy.sparse import diags

import importlib.util
from pathlib import Path
import modularity_vitality as mv_v2

<div style="font-size:14pt; font-weight:bold;">Importing Agent x Agent - All Communication</div>

In [2]:
agent_agent_all_comm = ig.read("/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/agent_agent_all_communication.graphml")

<div style="font-size:14pt; font-weight:bold;">Calculating Modularity Vitality Scores</div>

In [3]:
# Partioning the Network
part = agent_agent_all_comm.community_leiden(objective_function='modularity', weights='weight', resolution_parameter=1)
part.modularity

/tmp/ipykernel_25649/3847937952.py:2: DeprecationWarning: resolution_parameter keyword argument is deprecated, use resolution=... instead
  part = agent_agent_all_comm.community_leiden(objective_function='modularity', weights='weight', resolution_parameter=1)


0.5600532938211299

In [4]:
# Calculating Modularity Vitality Scores
vitalities = mv_v2.modularity_vitality(agent_agent_all_comm, part)

# Map to node identifiers
if "label" in agent_agent_all_comm.vs.attributes():
    names = agent_agent_all_comm.vs["label"]
elif "name" in agent_agent_all_comm.vs.attributes():
    names = agent_agent_all_comm.vs["name"]
else:
    names = list(range(agent_agent_all_comm.vcount()))

In [5]:
# Saving the Partition for Testing
partition_df = pd.DataFrame({
    "Node ID": pd.Series(names, dtype=str),
    "leiden_community": pd.Series(part.membership, dtype=int)
})

partition_path = "/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_partition_leiden_gamma1.csv"
os.makedirs(os.path.dirname(partition_path), exist_ok=True)
partition_df.to_csv(partition_path, index=False)
print(f"Saved partition → {partition_path}")

Saved partition → /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_partition_leiden_gamma1.csv


In [6]:
# Partition modularity at γ = 1
Q0 = float(part.modularity)
assert abs(Q0) > 0, "Partition modularity Q0 is zero; normalization undefined."

# Split into hub/bridge columns and build a tidy table
hub = [max(v, 0.0) for v in vitalities]
bridge = [max(-v, 0.0) for v in vitalities]

In [7]:
# Ensure arrays
vitalities = np.asarray(vitalities, dtype=float)
hub = np.maximum(vitalities, 0.0)
bridge = np.maximum(-vitalities, 0.0)

# Normalized columns (ORA-like)
hub_norm = np.where(hub > 0, hub / abs(Q0), 0.0)
bridge_norm = np.where(bridge > 0, bridge / abs(Q0), 0.0)

# (optional) denoise near-zero float chatter
eps = 1e-15
hub_norm = np.where(hub_norm < eps, 0.0, hub_norm)
bridge_norm = np.where(bridge_norm < eps, 0.0, bridge_norm)

# Sanity check lengths
n = agent_agent_all_comm.vcount()
assert len(vitalities) == n and len(names) == n and len(hub) == n and len(bridge) == n

In [8]:
# Construct Table
df = pd.DataFrame({
    "node": names,
    "vitality": vitalities,
    "raw_hub": hub,
    "raw_bridge": bridge,
    "hub_norm": hub_norm,
    "bridge_norm": bridge_norm,
    "community": part.membership  # 0-based community labels from igraph
})

df.head()

,node,vitality,raw_hub,raw_bridge,hub_norm,bridge_norm,community
0,828033366712688640,0.002914,0.002914,0.000000,0.005203,0.000000,0
1,1124877633206931456,-0.002779,0.000000,0.002779,0.000000,0.004961,0
2,8775672,0.003852,0.003852,0.000000,0.006877,0.000000,1
3,1448413205538148357,-0.002072,0.000000,0.002072,0.000000,0.003699,0
4,1163719671117303808,0.002877,0.002877,0.000000,0.005136,0.000000,2


In [9]:
results_path = "/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_vitality_results_python_gamma1.csv"
df.to_csv(results_path, index=False)
print(f"Saved vitality results → {results_path}")

Saved vitality results → /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_vitality_results_python_gamma1.csv


<div style="font-size:14pt; font-weight:bold;">Calculating Modularity Vitality Function Evaluation</div>

In [11]:
g = ig.read("/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/agent_agent_all_communication.graphml")

In [12]:
part = agent_agent_all_comm.community_leiden(objective_function='modularity', weights='weight', resolution=1)
membership = part.membership

In [13]:
def getSparseA(g):
    edges = [(e.source, e.target) for e in g.es()]
    sources, targets = list(zip(*edges))
    if g.is_weighted():
        weights = np.array(g.es['weight'], dtype=float)
    else:
        weights = np.ones(len(sources))
    self_loop_inds = (np.array(sources) == np.array(targets))
    weights[self_loop_inds] = weights[self_loop_inds] / 2
    weights = list(weights)
    A = csr_matrix((weights + weights, (sources + targets, targets + sources)),
                   shape=(g.vcount(), g.vcount()))
    return A

In [14]:
A = getSparseA(g)

In [15]:
from scipy.sparse import csr_matrix

# --- Hard map: Twitter ID (in g.vs['label']) -> handle (for readability) ---
id_to_handle = {
    "828033366712688640": "MyriadCsPhantom",
    "24112747": "INDOPACOM",
    "25930421": "US7thFleet",
    "18749026": "PACAF",
}

# --- Build A exactly like Julia getSparseA (halve loops, then symmetrize) ---
edges = [(e.source, e.target) for e in g.es]
sources, targets = ([], []) if not edges else list(zip(*edges))
sources = np.array(sources, dtype=int)
targets = np.array(targets, dtype=int)

weights = np.array(g.es["weight"], dtype=float) if "weight" in g.es.attributes() else np.ones(len(sources), dtype=float)
self_mask = (sources == targets)
w_halved = weights.copy()
w_halved[self_mask] *= 0.5

n = g.vcount()
A_dir = csr_matrix((w_halved, (sources, targets)), shape=(n, n))
A = A_dir + A_dir.T

# --- Build ID->index map using vertex attribute 'label' ---
labels = g.vs["label"]  # Twitter IDs as strings
ext2idx = {str(labels[i]): i for i in range(n)}

def idx_of(twitter_id_str):
    return ext2idx.get(str(twitter_id_str), None)

# --- Resolve indices for the four nodes ---
sent_id = "828033366712688640"
indopacom_id = "24112747"
us7_id = "25930421"
pacaf_id = "18749026"

sent_idx = idx_of(sent_id)
indo_idx = idx_of(indopacom_id)
us7_idx  = idx_of(us7_id)
pacaf_idx = idx_of(pacaf_id)

print("=== A checks with Twitter-ID mapping (label) ===")
for tid in [sent_id, indopacom_id, us7_id, pacaf_id]:
    i = idx_of(tid)
    handle = id_to_handle[str(tid)]
    if i is None:
        print(f"{handle} ({tid}): NOT FOUND")
    else:
        deg = float(A[i, :].sum())
        diag = float(A[i, i])
        print(f"{handle} ({tid}) idx={i} | degree={deg:.10f} | diag(A)={diag:.10f}")

# --- Sentinel neighbor probes (expect ~1.0 for each edge in this test) ---
if sent_idx is not None:
    checks = [
        ("INDOPACOM", indo_idx),
        ("PACAF", pacaf_idx),
        ("US7thFleet", us7_idx),
    ]
    for name, j in checks:
        if j is not None:
            print(f"A[MyriadCsPhantom, {name}] = {float(A[sent_idx, j]):.10f}")
        else:
            print(f"A[MyriadCsPhantom, {name}] = (neighbor not found)")
else:
    print("Sentinel not found; skipping neighbor probes.")

=== A checks with Twitter-ID mapping (label) ===
MyriadCsPhantom (828033366712688640) idx=0 | degree=3.0000000000 | diag(A)=0.0000000000
INDOPACOM (24112747) idx=17 | degree=926.0000000000 | diag(A)=3.0000000000
US7thFleet (25930421) idx=44 | degree=264.0000000000 | diag(A)=0.0000000000
PACAF (18749026) idx=46 | degree=222.0000000000 | diag(A)=1.0000000000
A[MyriadCsPhantom, INDOPACOM] = 1.0000000000
A[MyriadCsPhantom, PACAF] = 1.0000000000
A[MyriadCsPhantom, US7thFleet] = 1.0000000000


In [16]:
def getGroupIndicator(g, membership, rows=None):
    if not rows:
        rows = list(range(g.vcount()))
    cols = membership
    vals = np.ones(len(cols))
    group_indicator_mat = csr_matrix((vals, (rows, cols)),
                                     shape=(g.vcount(), max(membership) + 1))
    return group_indicator_mat

In [17]:
S = getGroupIndicator(g, membership, rows=None)

In [18]:
def getDegMat(node_deg_by_group, rows, cols):
    degrees = node_deg_by_group.sum(1)
    degrees = np.array(degrees).flatten()
    deg_mat = csr_matrix((degrees, (rows, cols)),
                         shape=node_deg_by_group.shape)
    degrees = degrees[:, np.newaxis]
    return degrees, deg_mat

In [19]:
from scipy.sparse import csr_matrix, diags, issparse

def _shape_str(x):
    if issparse(x):
        return f"{x.__class__.__name__} shape={x.shape} nnz={x.nnz}"
    elif isinstance(x, np.ndarray):
        return f"ndarray shape={x.shape} dtype={x.dtype}"
    else:
        return f"{type(x).__name__}"

def _vec_summary(name, v):
    v_np = np.asarray(v).ravel()
    print(f"{name}: {_shape_str(v_np)} | min={v_np.min():.12g} max={v_np.max():.12g} mean={v_np.mean():.12g}")

def _mat_summary(name, M):
    if issparse(M):
        print(f"{name}: {_shape_str(M)} sum={float(M.sum()):.12g}")
    else:
        M_np = np.asarray(M)
        print(f"{name}: {_shape_str(M_np)} sum={float(M_np.sum()):.12g}")

def newMods_test(g, part):
    # --- inputs / prelim ---
    weight_key = 'weight' if g.is_weighted() else None
    index      = list(range(g.vcount()))
    membership = np.array(part.membership, dtype=int)             # length n
    print(f"membership: {_shape_str(membership)} | n={membership.size} | min={membership.min()} max={membership.max()} uniq={len(np.unique(membership))}")

    # --- mass m ---
    m = float(np.sum(g.strength(weights=weight_key)) / 2.0)
    print(f"m (sum(A)/2) = {m:.12g}")

    # --- A ---
    A = getSparseA(g)                                             # expected: csr, symmetric
    _mat_summary("A", A)
    print(f"(A - A.T).nnz = {(A - A.T).nnz}")
    self_loops = float(A.diagonal().sum())
    print(f"self_loops (sum diag(A)) = {self_loops:.12g}")

    # --- S (group indicator) ---
    S = getGroupIndicator(g, membership, rows=index)              # expected: csr (n x C), one-hot
    _mat_summary("S (group_indicator_mat)", S)

    # --- node_deg_by_group = A * S ---
    node_deg_by_group = A * S                                     # expected: csr (n x C)
    _mat_summary("node_deg_by_group = A * S", node_deg_by_group)

    # --- internal_edges = (node_deg_by_group[index, membership].sum() + self_loops)/2 ---
    sel_vals = np.asarray(node_deg_by_group[index, membership]).ravel()   # (n,)
    internal_edges = float((sel_vals.sum() + self_loops) / 2.0)
    print(f"internal_edges = {internal_edges:.12g}")
    _vec_summary("node_deg_by_group[row, memb] (flattened)", sel_vals)

    # --- degrees, deg_mat ---
    degrees, deg_mat = getDegMat(node_deg_by_group, index, membership)
    degrees = np.asarray(degrees).ravel()                         # FORCE (n,)
    _vec_summary("degrees (from getDegMat)", degrees)
    _mat_summary("deg_mat (from getDegMat)", deg_mat)

    # --- node_deg_by_group += deg_mat ---
    node_deg_by_group = (node_deg_by_group + deg_mat).tocsr()
    _mat_summary("node_deg_by_group += deg_mat", node_deg_by_group)

    # --- group_degs = (deg_mat + diags(A.diagonal()) * S).sum(0) ---
    D = diags(A.diagonal())
    group_degs_mat = (deg_mat + D * S)                            # (n x C)
    _mat_summary("group_degs_mat (deg_mat + D*S)", group_degs_mat)
    group_degs = np.asarray(group_degs_mat.sum(0)).ravel()        # FORCE (C,)
    _vec_summary("group_degs = sum(0)", group_degs)

    # --- internal_deg = node_deg_by_group[index, membership] - degrees ---
    nd_bg_diag = np.asarray(node_deg_by_group[index, membership]).ravel()  # (n,)
    internal_deg = nd_bg_diag - degrees                                    # (n,)
    _vec_summary("internal_deg", internal_deg)

    # --- starCenter mask & safe denominators ---
    starCenter = (degrees == m)                                   # (n,) boolean
    print(f"starCenter: {_shape_str(starCenter)} | count={int(np.count_nonzero(starCenter))}")
    deg_safe = degrees.astype(float).copy()
    deg_safe[starCenter] = 0.0

    # --- q1_links ---
    denom_links = (m - deg_safe)                                  # (n,)
    _vec_summary("denom_links = (m - degrees)", denom_links)
    q1_links = (internal_edges - internal_deg) / denom_links      # (n,)
    _vec_summary("q1_links", q1_links)

    # --- expected_impact ---
    sum_group_degs_sq = float(np.sum(group_degs**2))              # scalar
    nd_bg_times_g = node_deg_by_group.dot(group_degs)             # (n,)
    if issparse(node_deg_by_group):
        nd_bg_sq_row = np.asarray(node_deg_by_group.multiply(node_deg_by_group).sum(axis=1)).ravel()
    else:
        nd_bg_sq_row = np.sum(np.power(node_deg_by_group, 2.0), axis=1)
    expected_impact = sum_group_degs_sq - 2.0 * nd_bg_times_g + nd_bg_sq_row  # (n,)
    print(f"sum(group_degs^2) = {sum_group_degs_sq:.12g}")
    _vec_summary("node_deg_by_group * group_degs^T", nd_bg_times_g)
    _vec_summary("rowwise sum(node_deg_by_group^2)", nd_bg_sq_row)
    _vec_summary("expected_impact", expected_impact)

    # --- q1_degrees ---
    denom_degrees = 4.0 * (m - deg_safe)**2                       # (n,)
    _vec_summary("denom_degrees = 4*(m - degrees)^2", denom_degrees)
    q1_degrees = expected_impact / denom_degrees                  # (n,)
    _vec_summary("q1_degrees", q1_degrees)

    # --- q1s and return ---
    q1s = q1_links - q1_degrees                                   # (n,)
    q1s[starCenter] = 0.0
    _vec_summary("q1s (final)", q1s)

    return q1s


In [20]:
q1s = newMods_test(g, part)

membership: ndarray shape=(1347,) dtype=int64 | n=1347 | min=0 max=143 uniq=144
m (sum(A)/2) = 4507
A: csr_matrix shape=(1347, 1347) nnz=6187 sum=8920
(A - A.T).nnz = 0
self_loops (sum diag(A)) = 94
S (group_indicator_mat): csr_matrix shape=(1347, 144) nnz=1347 sum=1347
node_deg_by_group = A * S: csr_matrix shape=(1347, 144) nnz=1796 sum=8920
internal_edges = 3432
node_deg_by_group[row, memb] (flattened): ndarray shape=(1347,) dtype=float64 | min=0 max=760 mean=5.02598366741
degrees (from getDegMat): ndarray shape=(1347,) dtype=float64 | min=0 max=926 mean=6.62212323682
deg_mat (from getDegMat): csr_matrix shape=(1347, 144) nnz=1347 sum=8920
node_deg_by_group += deg_mat: csr_matrix shape=(1347, 144) nnz=1796 sum=17840
group_degs_mat (deg_mat + D*S): csr_matrix shape=(1347, 144) nnz=1245 sum=9014
group_degs = sum(0): ndarray shape=(144,) dtype=float64 | min=0 max=3433 mean=62.5972222222
internal_deg: ndarray shape=(1347,) dtype=float64 | min=0 max=760 mean=5.02598366741
starCenter: ndar

In [22]:
print(q1s[:10])

[0.55689826 0.56258178 0.55597132 0.56187924 0.55695278 0.5611327
 0.55829816 0.554574   0.55861969 0.55858713]
